# dashboard view model v1

目标：验证 `src/dashboard` 能从统一 `dashboard_export.json` 构建页面交互所需的 view model，支持按模型、regime、window、fidelity 和 attribution feature 筛选。

In [1]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'SPEC.md').exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('cannot locate repo root from notebook cwd')
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.dashboard import DashboardViewFilters, load_dashboard_view_model

CARD = Path('research/dashboard_view_model_v1')
SOURCE_EXPORT = Path('research/schedule_dashboard_attribution_refresh_v1/dashboard_export.json')
CARD.mkdir(parents=True, exist_ok=True)

{'card': str(CARD), 'source_export': str(SOURCE_EXPORT)}

{'card': 'research/dashboard_view_model_v1',
 'source_export': 'research/schedule_dashboard_attribution_refresh_v1/dashboard_export.json'}

## 构建默认页面视图

In [2]:
default_view = load_dashboard_view_model(SOURCE_EXPORT)
(CARD / 'view_model_all.json').write_text(json.dumps(default_view, ensure_ascii=False, indent=2), encoding='utf-8')
{
    'schema_version': default_view['schema_version'],
    'model_card_count': len(default_view['model_cards']),
    'controls': default_view['controls'],
}

{'schema_version': 'dashboard.view_model.v1',
 'model_card_count': 5,
 'controls': {'models': ['gcn_mse__ic_weighted_composite',
   'gcn_mse__top_ic_factors',
   'ridge_mse__all_features',
   'ridge_mse__ic_weighted_composite',
   'ridge_mse__top_ic_factors'],
  'base_models': ['gcn_mse', 'ridge_mse'],
  'feature_sets': ['all_features', 'ic_weighted_composite', 'top_ic_factors'],
  'regimes': ['bull', 'downtrend', 'neutral', 'stress'],
  'windows': ['test'],
  'paper_fidelities': ['native_library', 'proxy_feature_adapter'],
  'attribution_features': ['soxl_soxx_relative_5d',
   'soxx_return_1d',
   'soxx_return_5d']}}

## 构建压力 regime + proxy graph 模型筛选视图

In [3]:
stress_proxy_view = load_dashboard_view_model(
    SOURCE_EXPORT,
    filters=DashboardViewFilters(
        models=('gcn_mse__top_ic_factors',),
        regimes=('stress',),
        windows=('test',),
        paper_fidelities=('proxy_feature_adapter',),
        attribution_features=('soxx_return_5d',),
    ),
)
(CARD / 'view_model_stress_proxy.json').write_text(
    json.dumps(stress_proxy_view, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
stress_proxy_view['model_cards'][0]

{'rank': 2,
 'model': 'gcn_mse__top_ic_factors',
 'base_model': 'gcn_mse',
 'feature_set': 'top_ic_factors',
 'return_pct': 51.71525641184807,
 'benchmark_return_pct': 62.1472468532841,
 'relative_return_pct': -10.431990441436035,
 'max_drawdown_pct': 26.174098020891567,
 'sharpe': 3.8156756348398457,
 'avg_position': 0.7497642874598264,
 'turnover': 2.1464782292222577,
 'stability_score': 52.861433775687544,
 'paper_fidelity': 'proxy_feature_adapter',
 'proxy_model': True,
 'promotion_blocker': 'Requires full paper-fidelity architecture replacement before production promotion.',
 'top_attribution_features': [{'model': 'gcn_mse',
   'feature': 'soxx_return_5d',
   'normalized_importance': 0.6224231933249113}],
 'regime_metrics': [{'model': 'gcn_mse__top_ic_factors',
   'regime': 'stress',
   'return_pct': 29.43452605190955,
   'benchmark_return_pct': 28.78548509716814,
   'relative_return_pct': 0.6490409547414089,
   'max_drawdown_pct': 5.6831838362351945,
   'sharpe': 11.0884337144475

## 写出本次分析摘要

In [4]:
stress_card = stress_proxy_view['model_cards'][0]
summary = {
    'schema_version': 'dashboard.view_model_evidence.v1',
    'status': 'completed',
    'source_export': str(SOURCE_EXPORT),
    'default_view_model': str(CARD / 'view_model_all.json'),
    'stress_proxy_view_model': str(CARD / 'view_model_stress_proxy.json'),
    'default_model_card_count': len(default_view['model_cards']),
    'available_controls': default_view['controls'],
    'filtered_model': stress_card['model'],
    'filtered_paper_fidelity': stress_card['paper_fidelity'],
    'filtered_proxy_model': stress_card['proxy_model'],
    'filtered_regime_rows': len(stress_card['regime_metrics']),
    'filtered_window_regime_rows': len(stress_card['window_regime_metrics']),
    'filtered_diagnostics': stress_card['diagnostics'][:5],
    'filtered_attribution_features': stress_card['top_attribution_features'],
    'risk_notes': [
        'view model 只读取 dashboard_export.json，不回读 artifacts 指向的 sidecar。',
        'gcn_mse 当前是 proxy_feature_adapter，页面要明确显示 fidelity 和 promotion blocker。',
    ],
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'dashboard.view_model_evidence.v1',
 'status': 'completed',
 'source_export': 'research/schedule_dashboard_attribution_refresh_v1/dashboard_export.json',
 'default_view_model': 'research/dashboard_view_model_v1/view_model_all.json',
 'stress_proxy_view_model': 'research/dashboard_view_model_v1/view_model_stress_proxy.json',
 'default_model_card_count': 5,
 'available_controls': {'models': ['gcn_mse__ic_weighted_composite',
   'gcn_mse__top_ic_factors',
   'ridge_mse__all_features',
   'ridge_mse__ic_weighted_composite',
   'ridge_mse__top_ic_factors'],
  'base_models': ['gcn_mse', 'ridge_mse'],
  'feature_sets': ['all_features', 'ic_weighted_composite', 'top_ic_factors'],
  'regimes': ['bull', 'downtrend', 'neutral', 'stress'],
  'windows': ['test'],
  'paper_fidelities': ['native_library', 'proxy_feature_adapter'],
  'attribution_features': ['soxl_soxx_relative_5d',
   'soxx_return_1d',
   'soxx_return_5d']},
 'filtered_model': 'gcn_mse__top_ic_factors',
 'filtered_

## 结论

- 做了什么：从 `dashboard_export.json` 构建 `dashboard.view_model.v1`，输出默认页面视图和一个 stress/proxy/attribution 筛选视图。
- 核心指标：默认视图的模型卡片数量、可用 controls、筛选视图的模型、fidelity、window-regime 行数以 `analysis_summary.json` 为准。
- regime 差异：筛选视图保留 stress regime 下的 `regime_metrics`、`window_regime_metrics` 和 `model_effect_deltas`，页面可以直接展示失效段诊断。
- 是否晋升：这是 dashboard 交互数据层验证，不是模型晋升。
- 风险：view model 不负责重新计算回测，只消费已生成 export；上游 export 的 top_n 会限制页面可见模型集合。
- 下一步：补 dashboard 前端或静态页面读取 `dashboard.view_model.v1`，增加 model/regime/window/fidelity/attribution 控件。